# MEFF v3 — train ONE model on Kaggle

**Before running:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**.
Input: only the dataset with the 5,955 images. The code and the random 60:20:20 split are built into this notebook.

Edit only the config cell, then **Save Version → Save & Run All (Commit)**.
One notebook = one model. Six models in total:

| Account | Notebook 1 | Notebook 2 |
|---|---|---|
| A | `meff`, seed 42 (+ `RUN_COMPLEXITY = True`) | `dsg`, seed 42 |
| B | `meff`, seed 43 | `dsg`, seed 43 |
| C | `meff`, seed 44 | `dsg`, seed 44 |


In [ ]:
# ===== EDIT ONLY THIS CELL =====
VARIANT = "meff"        # "meff" or "dsg"
SEED = 42               # 42, 43 or 44
RUN_COMPLEXITY = False  # True in ONE notebook only (account A, meff, seed 42)
# ================================
MAX_HOURS = 11.0        # Kaggle stops GPU sessions at 12 h; training saves and exits before that
SMOKE_TEST = False      # True = 1-epoch pipeline test on 16 images/class (writes to runs_smoke/, not valid for results)

In [ ]:
# DO NOT EDIT - embedded code package (meff_v3.py + frozen random split), base64 zip
PKG_B64 = "..."

In [ ]:
import base64, io
import os, glob, shutil, subprocess, sys, json, zipfile, hashlib
assert VARIANT in ("meff", "dsg") and SEED in (42, 43, 44), "check VARIANT / SEED"
W = "/kaggle/working"
# --- embedded package (code + frozen random 60:20:20 split) ---
_buf = base64.b64decode(PKG_B64)
assert hashlib.md5(_buf).hexdigest() == "39aaf9f55a307fcb79fd7efaa659094b", "embedded package damaged"
if not os.path.exists(f"{W}/MEFF_v3/meff_v3.py"):
    zipfile.ZipFile(io.BytesIO(_buf)).extractall(W)
for _line in open(f"{W}/MEFF_v3/CHECKSUMS.md5"):
    _h, _f = _line.split()
    assert hashlib.md5(open(f"{W}/MEFF_v3/{_f}", "rb").read()).hexdigest() == _h, f"checksum mismatch: {_f}"
PKG = "embedded (package md5 39aaf9f55a307fcb79fd7efaa659094b)"
print("package checksums OK")
CLS = ["NONE", "INFECTION", "ISCHAEMIA", "BOTH"]
cands = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ISCHAEMIA", recursive=True)
                if all(os.path.isdir(os.path.join(os.path.dirname(p), k)) for k in CLS)})
assert len(cands) == 1, f"dataset folder with NONE/INFECTION/ISCHAEMIA/BOTH not found exactly once: {cands}"
DATA = cands[0]
RUN = f"{VARIANT}_s{SEED}"
RUNS = "runs_smoke" if SMOKE_TEST else "runs"
os.chdir(f"{W}/MEFF_v3")
print("package :", PKG)
print("data    :", DATA, {k: len(os.listdir(os.path.join(DATA, k))) for k in CLS})
print("run     :", RUN, "->", f"{W}/MEFF_v3/{RUNS}/{RUN}")
print("code md5:", hashlib.md5(open("meff_v3.py", "rb").read()).hexdigest())

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0); arch = f"sm_{cap[0]}{cap[1]}"
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0), arch, "| built for", torch.cuda.get_arch_list())
assert arch in torch.cuda.get_arch_list(), f"This PyTorch build cannot use {arch} (e.g. P100). Choose GPU T4 x2."


In [ ]:
r = subprocess.run([sys.executable, "meff_v3.py", "verify", "--data_dir", DATA, "--split_dir", "split"], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "VERIFY FAILED - the images do not match the frozen split. Stop and report."


In [ ]:
import torchvision
torchvision.models.densenet169(weights="IMAGENET1K_V1")
print("ImageNet weights OK (Internet must be On)")

In [ ]:
# Resume: if an earlier version of THIS notebook stopped at the time limit, attach its output
# (Add Input -> Your Work -> this notebook) and this cell copies runs/<RUN>/ back so training continues.
dst = f"{W}/MEFF_v3/{RUNS}/{RUN}"
prev = [p for p in glob.glob(f"/kaggle/input/**/{RUNS}/{RUN}/last.pth", recursive=True)]
prev_done = [p for p in glob.glob(f"/kaggle/input/**/{RUNS}/{RUN}/DONE", recursive=True)]
if prev_done:
    print("An attached earlier version is already DONE:", prev_done[0], "- nothing to train. Download its results instead.")
elif prev and not os.path.exists(f"{dst}/last.pth"):
    shutil.copytree(os.path.dirname(prev[0]), dst, dirs_exist_ok=True)
    print("RESUMING from", prev[0])
else:
    print("fresh start" if not os.path.exists(dst) else "folder exists in this session - continuing")

In [ ]:
args = ["train", "--variant", VARIANT, "--seed", str(SEED), "--data_dir", DATA, "--split_dir", "split",
        "--out_dir", f"{RUNS}/{RUN}", "--max_hours", str(MAX_HOURS)]
args += ["--workers", "4"]
if SMOKE_TEST:
    args += ["--smoke", "--smoke_epochs", "1", "--smoke_batch", "32"]
print("python meff_v3.py", " ".join(args), flush=True)
p = subprocess.Popen([sys.executable, "-u", "meff_v3.py", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
rc = p.wait()
assert rc == 0, f"training exited with code {rc} - see the log above"


In [ ]:
d = f"{W}/MEFF_v3/{RUNS}/{RUN}"
if os.path.exists(f"{d}/DONE"):
    meta = json.load(open(f"{d}/run_meta.json"))
    out = f"{W}/{RUN}_results{'_SMOKE' if SMOKE_TEST else ''}.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in ["DONE", "run_meta.json", "history.csv", "val_predictions.csv", "test_predictions.csv"]:
            z.write(f"{d}/{f}", f"{RUN}/{f}")
    print("DONE. Download from the Output tab:", os.path.basename(out))
    print({k: meta[k] for k in ["variant", "seed", "gpu", "torch", "best_epoch", "best_val_macro_f1", "epochs_run", "pretrained", "smoke"]})
else:
    print("NOT DONE (time limit reached). Save this version. Then: New version -> Add Input -> Your Work -> this notebook's"
          " output -> Save & Run All. The resume cell continues from last.pth.")

In [ ]:
if RUN_COMPLEXITY and not SMOKE_TEST:
    r = subprocess.run([sys.executable, "meff_v3.py", "complexity", "--out_dir", f"{W}/complexity"], capture_output=True, text=True)
    print(r.stdout, r.stderr[-1500:])
    print("Download complexity/complexity_v3.csv from the Output tab.")
else:
    print("complexity skipped")